# Human Blind Review of the Handling Validation Sample

The 200 cases in `derived/validation-labeled.csv` carry two independent labels:

- `label`: assigned by an AI language model (Claude, Anthropic) under the written standard in the README.
- `human_label`: assigned by the author, blind to the AI labels and to the rule's predictions, under the same written standard.

This notebook measures (1) agreement between the two labelers and (2) the accuracy of the frozen handling rule against each set of labels. It reads only the published file and needs no download from OSHA.

In [1]:
import os, re, pandas as pd, numpy as np
URL = ('https://raw.githubusercontent.com/yuxuan-huang-data/us-care-safety-signals/'
       'main/nursing-facility-handling/derived/validation-labeled.csv')
v = pd.read_csv('derived/validation-labeled.csv' if os.path.exists('derived/validation-labeled.csv') else URL)
v['ai'] = v.label.str.strip(); v['human'] = v.human_label.str.strip()
print(len(v), 'cases'); print(v[['ai', 'human']].apply(pd.Series.value_counts))

200 cases
              ai  human
other         88     89
care_general  66     66
handling      46     45


## 1. Agreement between the two labelers

In [2]:
def kappa(a, b):
    cats = sorted(set(a) | set(b)); po = (a == b).mean()
    pe = sum((a == c).mean() * (b == c).mean() for c in cats)
    return (po - pe) / (1 - pe)
hb, ab = v.human == 'handling', v.ai == 'handling'
agree = pd.DataFrame({
    'agreement': [round((hb == ab).mean(), 3), round((v.human == v.ai).mean(), 3)],
    'cohen_kappa': [round(kappa(hb, ab), 3), round(kappa(v.human, v.ai), 3)]},
    index=['handling vs not handling', 'three categories'])
print(pd.crosstab(v.ai, v.human, rownames=['AI'], colnames=['human'], margins=True)); agree

human         care_general  handling  other  All
AI                                              
care_general            45         6     15   66
handling                 9        37      0   46
other                   12         2     74   88
All                     66        45     89  200


,agreement,cohen_kappa
handling vs not handling,0.915,0.758
three categories,0.780,0.658


## 2. The frozen rule against each reference

In [3]:
PERSON   = r"\b(?:resident|residents|patient|patients|pt|res)\b"
MOVEMENT = (r"\b(?:transfer\w*|lift\w*|reposition\w*|boost\w*|turn\w*|roll\w*|"
            r"pull\w* (?:up|over)|pivot\w*|hoyer|mechanical lift|gait belt|sit[- ]to[- ]stand|"
            r"slide sheet|draw sheet|bed to chair|chair to bed|wheelchair to bed|bed to wheelchair|"
            r"toilet\w* transfer|ambulat\w*)\b")
PHRASE   = r"patient handling|resident handling|resident care transfer"
MOVE_BROAD = (r"\b(?:mov(?:e|ed|es|ing)|catch\w*|caught|lower(?:ed|ing)?|"
              r"into (?:the |her |his |their )?(?:bed|chair|wheelchair|shower chair)|up in bed|"
              r"dead ?weight|weight of (?:the )?(?:resident|patient)|"
              r"(?:pull|push)\w* (?:the )?(?:resident|patient)|"
              r"pick\w* (?:him|her|them) up|pick\w* up (?:the )?(?:resident|patient))\b")
n = v.narr.astype(str).str.lower(); p = n.str.contains(PERSON)
strict = (p & n.str.contains(MOVEMENT)) | n.str.contains(PHRASE)
broad = strict | (p & n.str.contains(MOVE_BROAD))
rows = []
for ref_name, ref in [('AI labels', ab), ('human labels', hb)]:
    for rule_name, r in [('strict', strict), ('broad', broad)]:
        tp = int((r & ref).sum())
        rows.append({'reference': ref_name, 'rule': rule_name, 'reference positive': int(ref.sum()),
                     'precision': round(tp / r.sum(), 3), 'recall': round(tp / ref.sum(), 3)})
res = pd.DataFrame(rows)
k, N, z = int(hb.sum()), len(v), 1.96; ph = k / N
c = (ph + z*z/(2*N)) / (1 + z*z/N); hw = z*np.sqrt(ph*(1-ph)/N + z*z/(4*N*N)) / (1 + z*z/N)
print(f'human-labeled share handling: {ph:.3f}  95% CI [{c-hw:.3f}, {c+hw:.3f}]')
res

human-labeled share handling: 0.225  95% CI [0.173, 0.288]


,reference,rule,reference positive,precision,recall
0,AI labels,strict,46,0.846,0.717
1,AI labels,broad,46,0.698,0.804
2,human labels,strict,45,0.846,0.733
3,human labels,broad,45,0.698,0.822


## 3. Disagreements on handling vs not handling

In [4]:
d = v[hb != ab][['row', 'ai', 'human', 'narr']]
d['narr'] = d.narr.str[:140]
pd.set_option('display.max_colwidth', 150); pd.set_option('display.width', 220)
d

,row,ai,human,narr
4,6354,other,handling,moving lift in parking lot at an incline. lift rolled over left foot. crush injury left toe lift
25,4314,handling,care_general,ee was changing a resident ee was changing a resident when she felt a pop in her back pain to back ee was assisting resident alone and resid
35,19814,handling,care_general,changing or either tansferring a patient ee reports she hit her [redacted] a mounted tv when changing a resident. contusion skull furnituref
40,11449,care_general,handling,getting ready to help resident to bed resident scratched employee scratch that broke skin on upper left arm resident
48,5058,care_general,handling,helping with cares on a resident. assisting with resident cares when felt a pop in lower back. strain in lower back. n a
50,4445,handling,care_general,assisting resident resident fell back on employees wrist when helping him off the floor sprain to left wrist patient
78,12755,handling,care_general,making rounds and found resident on the floor employee was assisting resident up off floor when resident leaned over and bite her on the sto
87,19081,care_general,handling,loading a resident into transport. the employee loaded the resident onto the lift locked the wheelchair into place and then attempted to tak
98,23142,care_general,handling,performing patient care employee reported on voicemail she 'd been injured by a resident. upon phone call- states resident kicked her in the
100,1507,handling,care_general,associate heard someone in [redacted] yelling and associate went into room and resident was falling and associate caught resident associate


In [5]:
os.makedirs('derived', exist_ok=True)
pd.concat([agree.reset_index(names='measure')]).to_csv('derived/human-review-agreement.csv', index=False)
res.to_csv('derived/human-review-rule-accuracy.csv', index=False)
print(sorted(os.listdir('derived')))

['human-review-agreement.csv', 'human-review-rule-accuracy.csv', 'validation-labeled.csv']
